[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_dae_full.ipynb)

# Part A: the final DAE on all the images

The best DAE so far is still the final DAE of search 2 (`config.py`: salt and pepper 0.5, h 256, conv 32-64): val k-NN 0.0994, against 0.0938 for the winner of the 10% search (patch_mask 0.5 with dropout 0.3). On 10% of the images salt and pepper lost in the first grid, so dropout and the learning rate were never tried with it.

Its val k-NN peaks at epoch 5 and then falls while the loss keeps falling; with patch_mask, dropout 0.3 kept the curve flat. So here, on all the images for 20 epochs, by **val k-NN top-1**:

- **dropout** 0, 0.1, 0.2, 0.3 × **learning rate** 1e-3, 5e-4: 8 runs, about 1.5 hours on an L4.
- The best of them against the winner of the 10% search (read from Drive). The best becomes `"dae"` in `tuning_choice.json`; the earlier choice is kept as `"dae_search3"`.

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU.

In [1]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 3 (delta 1), reused 1 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 75.49 KiB | 5.39 MiB/s, done.
From https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR
   20e8c40..fe4cab6  main       -> origin/main
Updating 20e8c40..fe4cab6
Fast-forward
 colab_dae_grids_2.ipynb | 4651 ++++++++++++++++++++++++++++++++++++++++++++++-
 1 file changed, 4634 insertions(+), 17 deletions(-)
fe4cab6 (HEAD -> main, origin/main, origin/HEAD) colab_dae_grids_2.ipynb with the outputs of grids 6-7 and stage 2: dropout 0.3 wins (val k-NN 0.0938)


In [2]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

NVIDIA L4 | 12 CPU cores


## Drive
Every run writes to `midterm_runs`, next to the earlier runs. The runs here have `tune_pretrain_DAE` in their folder name.

In [3]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_runs"
assert os.path.isdir(OUT), f"no folder {OUT}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## TensorBoard
Filter the runs with `tune_pretrain`, then *Toggle all runs*. `epoch/val_knn_top1` shows where each run peaks, `epoch/val_loss` the reconstruction, the TEXT tab each run's `run_info`.

In [4]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

Reusing TensorBoard on port 6006 (pid 1916), started 10:06:14 ago. (Use '!kill 1916' to kill it.)

<IPython.core.display.Javascript object>

## The runs
`train` runs train.py only if the run has no `results.json` yet, so after a disconnect a finished run is read from Drive and an unfinished one continues from its `last.ckpt`.

In [5]:
import importlib
import itertools
import json
import sys

import pandas as pd

RUN = "--mode pretrain --tune --num_epochs 20"  # all the images, the full k-NN bank, val only (no test)
CHOICE_FILE = f"{OUT}/tuning_choice.json"
# The final DAE of search 2 (config.py)
base = {"noise_type": "salt_pepper", "noise_level": 0.5, "patch_size": 8, "conv1": 32, "conv2": 64,
        "latent_dim": 256, "dropout": 0.0, "augment": "flip", "batch_size": 512, "optimizer": "adamw",
        "activation": "gelu", "lr": 0.001, "wd": 0.01, "scheduler": "none", "noise_type2": "none",
        "noise_level2": 0.0}


def flags_of(params):
    """The train.py flags of one DAE."""
    p = dict(params)
    return f"--channels {p.pop('conv1')} {p.pop('conv2')} " + " ".join(f"--{k} {v}" for k, v in p.items())


def folder_of(flags):
    """The run folder train.py uses for these flags (run_info.folder_name)."""
    import config, utils, run_info
    for module in (config, utils, run_info):
        importlib.reload(module)
    sys.argv = ["train.py", *flags.split(), "--output_dir", OUT]
    return f"{OUT}/{run_info.folder_name(utils.parse_args().mode)}"


def train(flags):
    """One run of train.py (skipped if it finished before): its val numbers and its folder."""
    if not os.path.exists(f"{folder_of(flags)}/results.json"):
        !python train.py {flags} --output_dir "{OUT}"
    folder = folder_of(flags)
    with open(f"{folder}/results.json") as f:
        return {**json.load(f), "run": os.path.basename(folder)}

## Dropout × learning rate
Every combination, the rest as in `base`: 4 × 2 = 8 runs. Dropout 0 with lr 1e-3 is the final DAE of search 2 again, in a tune run.

In [6]:
rows = []
for dropout, lr in itertools.product([0.0, 0.1, 0.2, 0.3], [0.001, 0.0005]):
    params = {**base, "dropout": dropout, "lr": lr}
    r = train(f"{RUN} {flags_of(params)}")
    rows.append({**params, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
grid = pd.DataFrame(rows).sort_values("val_knn", ascending=False)
display(grid[["dropout", "lr", "val_knn", "val_loss"]].round(4))

Seed set to 42
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
`Trainer(limit_test_batches=1.0)` was configured so 100% of the batches will be used..
DenoisingAutoEncoder(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): GELU(approximate='none')
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): GELU(approximate='none')
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): Linear(in_features=16384, out_features=256, b

,dropout,lr,val_knn,val_loss
6,0.3,0.0010,0.1058,0.2289
4,0.2,0.0010,0.1045,0.2256
3,0.1,0.0005,0.1020,0.2375
2,0.1,0.0010,0.1019,0.2816
7,0.3,0.0005,0.0991,0.2387
0,0.0,0.0010,0.0990,0.2401
5,0.2,0.0005,0.0988,0.2793
1,0.0,0.0005,0.0938,0.2694


## The choice
The 8 runs against the winner of the 10% search (trained in `colab_dae_grids_2.ipynb`, read from Drive). The best by val k-NN is saved as `"dae"`.

In [7]:
choices = json.load(open(CHOICE_FILE))
choices.setdefault("dae_search3", choices["dae"])  # the choice of the 10% search, kept
search3 = {k: v for k, v in choices["dae_search3"].items() if k != "num_epochs"}
r = train(f"{RUN} {flags_of(search3)}")
finals = pd.DataFrame([*rows, {**search3, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]}])
finals = finals.drop_duplicates(subset="run").sort_values("val_knn", ascending=False)
display(finals[["noise_type", "dropout", "lr", "val_knn", "val_loss", "run"]].round(4))

choice = {**finals[list(base)].to_dict("records")[0], "num_epochs": 20}
choices["dae"] = choice
with open(CHOICE_FILE, "w") as f:
    json.dump(choices, f, indent=1)
print("the DAE:", choice)

,noise_type,dropout,lr,val_knn,val_loss,run
6,salt_pepper,0.3,0.0010,0.1058,0.2289,date24112026_time112843_CNN3_tune_pretrain_DAE...
4,salt_pepper,0.2,0.0010,0.1045,0.2256,date24112026_time110557_CNN3_tune_pretrain_DAE...
3,salt_pepper,0.1,0.0005,0.1020,0.2375,date24112026_time105438_CNN3_tune_pretrain_DAE...
2,salt_pepper,0.1,0.0010,0.1019,0.2816,date24112026_time104320_CNN3_tune_pretrain_DAE...
7,salt_pepper,0.3,0.0005,0.0991,0.2387,date24112026_time114002_CNN3_tune_pretrain_DAE...
0,salt_pepper,0.0,0.0010,0.0990,0.2401,date24112026_time102042_CNN3_tune_pretrain_DAE...
5,salt_pepper,0.2,0.0005,0.0988,0.2793,date24112026_time111724_CNN3_tune_pretrain_DAE...
1,salt_pepper,0.0,0.0005,0.0938,0.2694,date24112026_time103201_CNN3_tune_pretrain_DAE...
8,patch_mask,0.3,0.0010,0.0938,0.2196,date24112026_time021632_CNN3_tune_pretrain_DAE...


the DAE: {'noise_type': 'salt_pepper', 'noise_level': 0.5, 'patch_size': 8, 'conv1': 32, 'conv2': 64, 'latent_dim': 256, 'dropout': 0.3, 'augment': 'flip', 'batch_size': 512, 'optimizer': 'adamw', 'activation': 'gelu', 'lr': 0.001, 'wd': 0.01, 'scheduler': 'none', 'noise_type2': 'none', 'noise_level2': 0.0, 'num_epochs': 20}
